# Phase 3 RF-QRC tail probe

This notebook is the lightweight runner for `scripts/run_phase3_rf_qrc_tail_probe.py`.

Purpose: test whether an RF-QRC-style second-encoding feature map reduces the Phase 2 failure mode: under-dispersion and q90/q95 tail compression.

Decision gate: keep this path only if the second-encoding variant improves test-set prediction std, q90/q95 F1, top-bin predicted/actual ratio, or effective feature rank over the single-encoding control.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

print(PROJECT_ROOT)


## Run the controlled probe

Use `--data-path` if the automatic dataset search does not find the Phase 2 modeling table.


In [ ]:
from run_phase3_rf_qrc_tail_probe import main

main([
    '--input-mode', 'level_rate',
    '--leak', '0.3',
    '--ridge-alpha', '3000',
])


## Inspect test summary


In [ ]:
import pandas as pd

summary_path = PROJECT_ROOT / 'results' / 'tables' / 'phase3_rf_qrc_tail_probe_test_summary_level_rate.csv'
summary = pd.read_csv(summary_path)
cols = [
    'run_name', 'rmse', 'corr', 'actual_std', 'pred_std',
    'q90_f1', 'q90_precision', 'q90_recall',
    'q95_f1', 'q95_precision', 'q95_recall',
    'top20_pred_actual_ratio', 'effective_rank',
]
summary[[c for c in cols if c in summary.columns]]


## Decision rule

The RF-QRC-style path is worth carrying forward only if `rf_qrc_second_encode_level_rate` improves at least two tail-bottleneck metrics over `rf_qrc_single_encode_level_rate`, without destroying correlation. Otherwise, drop this gate-style feature-map path and return to the matched classical-spin / Bloqade level-rate Rydberg path.
